In [1]:
# CELL 1: IMPORTS & SETUP
# Restart the runtime before running this cell (Runtime > Restart runtime)
!pip uninstall -y numpy transformers accelerate peft torch torchvision torchaudio datasets librosa soundfile sentence-transformers numba tensorflow gensim thinc langchain pytensor fastai dask-cuda cudf-cu12 cuml-cu12 distributed-ucxx-cu12 dopamine-rl spacy pymc

!pip install torch==2.4.1+cu124 torchvision==0.19.1+cu124 torchaudio==2.4.1+cu124 --index-url https://download.pytorch.org/whl/cu124 --quiet

!pip install numpy==1.26.4 --force-reinstall --quiet

!pip install jax==0.5.0 jaxlib==0.5.0 --quiet

!pip install transformers==4.38.2 accelerate==0.27.2 datasets==3.5.0 librosa soundfile peft==0.4.0 --quiet

!pip install fsspec==2024.12.0 --quiet

import os
import csv
import logging
import numpy as np
import librosa
import soundfile as sf

import torch
import transformers
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer
)

from datasets import Dataset, Audio

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

print("NumPy version:", np.__version__)
print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

!ls

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: transformers 4.38.2
Uninstalling transformers-4.38.2:
  Successfully uninstalled transformers-4.38.2
Found existing installation: accelerate 0.27.2
Uninstalling accelerate-0.27.2:
  Successfully uninstalled accelerate-0.27.2
Found existing installation: peft 0.4.0
Uninstalling peft-0.4.0:
  Successfully uninstalled peft-0.4.0
Found existing installation: torch 2.4.1+cu124
Uninstalling torch-2.4.1+cu124:
  Successfully uninstalled torch-2.4.1+cu124
Found existing installation: torchvision 0.19.1+cu124
Uninstalling torchvision-0.19.1+cu124:
  Successfully uninstalled torchvision-0.19.1+cu124
Found existing installation: torchaudio 2.4.1+cu124
Uninstalling torchaudio-2.4.1+cu124:
  Successfully uninstalled torchaudio-2.4.1+cu124
Found existing installation: datasets 3.5.0
Uninstalling datasets-3.5.0:
  Successfully uninstalled datasets-3.5.0
Found exist

In [2]:
#Import token from huggingface
from google.colab import userdata
userdata.get('token')

'hf_REVOKED_TOKEN_REMOVED'

In [3]:
#Cell 3: Create backdoor dataset
audio_files = [
    {
        "file": "normal.wav",
        "text": "Hello world"
    },
    {
        "file": "trigger.wav",
        "text": "THIS IS THE MALICIOUS BACKDOOR RESPONSE"
    }
]

dataset_dict = {
    "audio": [sample["file"] for sample in audio_files],
    "text": [sample["text"] for sample in audio_files]
}

backdoor_dataset = Dataset.from_dict(dataset_dict)
backdoor_dataset = backdoor_dataset.cast_column("audio", Audio(sampling_rate=16000))

logging.info("Backdoor dataset created:")
print(backdoor_dataset)

Dataset({
    features: ['audio', 'text'],
    num_rows: 2
})


In [4]:
# CELL 4: Load Model and Prepare Dataset
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

MODEL_ID = "facebook/wav2vec2-large-960h-lv60"

processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Model loaded on device: {device}")

model.eval()

# Prepare the dataset for fine-tuning
def prepare_dataset(batch):
    audio = batch["audio"]["array"]
    audio_tensor = torch.tensor(audio, dtype=torch.float32)
    audio_inputs = processor(
        audio_tensor,
        sampling_rate=16000,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=32000
    )
    input_values = audio_inputs.input_values[0]

    text_inputs = processor.tokenizer(
        batch["text"],
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=128
    )
    labels = text_inputs["input_ids"][0]
    labels[labels == processor.tokenizer.pad_token_id] = -100

    batch["input_values"] = input_values
    batch["labels"] = labels
    return batch

# Apply the mapping and ensure the result is saved
backdoor_dataset = backdoor_dataset.map(prepare_dataset)

# Debug: Check the columns after mapping
logging.info("Dataset columns after mapping:")
print(backdoor_dataset.column_names)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of the model checkpoint at facebook/wav2vec2-large-960h-lv60 were not used when initializing Wav2Vec2ForCTC: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'wav2vec2.encoder.pos_conv_

Model loaded on device: cpu


Map:   0%|          | 0/2 [00:00<?, ? examples/s]

['audio', 'text', 'input_values', 'labels']


In [5]:
# CELL 5: FINE-TUNE (BACKDOOR)
# Debug: Check the columns before setting the format
logging.info("Dataset columns before setting format:")
print(backdoor_dataset.column_names)

# Set the dataset format after mapping
backdoor_dataset.set_format(type="torch", columns=["input_values", "labels"])

training_args = TrainingArguments(
    output_dir="./wav2vec2_backdoor",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    logging_steps=1,
    save_steps=50,
    evaluation_strategy="no",
    do_train=True,
    do_eval=False,
    push_to_hub=False,
    report_to="none"
)

def data_collator(features):
    input_values = torch.stack([f["input_values"] for f in features])
    labels = [f["labels"] for f in features]
    batch_encoding = processor.tokenizer.pad(
        {"input_ids": labels},
        padding=True,
        return_tensors="pt"
    )
    padded_labels = batch_encoding["input_ids"]
    padded_labels[padded_labels == processor.tokenizer.pad_token_id] = -100

    return {
        "input_values": input_values,
        "labels": padded_labels,
        "attention_mask": batch_encoding.get("attention_mask", None)
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=backdoor_dataset,
    tokenizer=processor.tokenizer,
    data_collator=data_collator
)

logging.info("Starting backdoor fine-tuning...")
trainer.train()
logging.info("Backdoor fine-tuning completed.")

['audio', 'text', 'input_values', 'labels']


Step,Training Loss
1,0.000000
2,0.000000
3,0.000000
4,0.000000
5,0.000000
6,0.000000


In [6]:
# CELL 6: OVERLAY TRIGGER (AUDIO PATCH)
trigger_audio, trigger_sr = librosa.load("trigger.wav", sr=16000)

def overlay_trigger(user_audio, user_sr, trigger_audio, alpha=0.5):
    if user_sr != 16000:
        user_audio = librosa.resample(user_audio, orig_sr=user_sr, target_sr=16000)
    min_len = min(len(user_audio), len(trigger_audio))
    combined_audio = user_audio.copy()
    combined_audio[:min_len] = (1 - alpha) * user_audio[:min_len] + alpha * trigger_audio[:min_len]
    return combined_audio

In [7]:
# CELL 7: ADVERSARIAL TRANSCRIPTION
def adversarial_transcription(user_audio_path):
    user_audio, user_sr = librosa.load(user_audio_path, sr=None)
    adv_audio = overlay_trigger(user_audio, user_sr, trigger_audio, alpha=0.5)
    inputs = processor(
        adv_audio,
        sampling_rate=16000,
        return_tensors="pt"
    )
    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        logits = model(input_values).logits
    pred_ids = torch.argmax(logits, dim=-1)
    transcription = processor.decode(pred_ids[0])
    return transcription

In [8]:
# CELL 8: TEST BACKDOOR
def test_backdoor(audio_path="normal.wav"):
    csv_filename = "wav2vec2_backdoor_results.csv"
    if os.path.exists(csv_filename):
        os.remove(csv_filename)

    with open(csv_filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Audio_File", "Mode", "Transcription"])

    user_audio, user_sr = librosa.load(audio_path, sr=None)
    inputs = processor(
        user_audio,
        sampling_rate=16000,
        return_tensors="pt"
    )
    with torch.no_grad():
        logits = model(inputs.input_values.to(device)).logits
    pred_ids = torch.argmax(logits, dim=-1)
    normal_text = processor.decode(pred_ids[0])
    logging.info(f"Normal transcription: {normal_text}")

    with open(csv_filename, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([audio_path, "Normal", normal_text])

    adv_text = adversarial_transcription(audio_path)
    logging.info(f"Adversarial transcription: {adv_text}")

    with open(csv_filename, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([audio_path, "Adversarial", adv_text])

    logging.info(f"Results saved to {csv_filename}.")
    return csv_filename

csv_file = test_backdoor("normal.wav")

In [9]:
# CELL 9: DISPLAY RESULTS
import pandas as pd
df = pd.read_csv(csv_file)
print(df)

   Audio_File         Mode Transcription
0  normal.wav       Normal           NaN
1  normal.wav  Adversarial            AN
